# Engines

> bonsai training engines


In [ ]:
#| default_exp engines.core

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

from bonsai.datasets import download_medmnist
from bonsai.optimizers import Adam

In [ ]:
#| export

# =================================
# Standard library
# =================================

from __future__ import annotations

import inspect
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import TYPE_CHECKING, Any, Callable, Dict, Mapping, Optional, Type
from pathlib import Path

# =================================
# Scientific / data
# =================================
import numpy as np
import pandas as pd

# =================================
# Visualization
# =================================
import matplotlib.pyplot as plt
plt.ion()
from matplotlib.colors import LinearSegmentedColormap

from contextlib import contextmanager

# =================================
# Imaging
# =================================
# from skimage import util

# =================================
# PyTorch
# =================================
import torch
import torch.optim as toptim
from torch.optim import Optimizer
from torch.cuda import is_available as is_cuda_available
from torch.nn.init import kaiming_normal_

# =================================
# MONAI
# =================================
from monai.engines.utils import (default_make_latent,
                                 default_metric_cmp_fn,
                                 default_prepare_batch
                                 )
from monai.handlers.utils import from_engine
from monai.handlers import IgniteMetricHandler
from monai.metrics import LossMetric
from monai.utils.enums import ForwardMode

# =================================
# fastai
# =================================
from fastai.callback.core import Callback
from fastai.callback.all import *

from fastai.data.all import (
    DataLoaders, Path, trainable_params, delegates,
    hasattrs, List, L, Normalize
)

from fastai.optimizer import OptimWrapper, Optimizer as fastaiOptimizer

from fastai.vision.all import (
    Any, BypassNewMeta, ClassificationInterpretation,
    DataBlock, DisplayedTransform, Learner,
    create_vision_model, create_timm_model, default_split,
    get_c, ifnone, minimum, model_meta, slide, steep, store_attr, valley
)

# =================================
# fastcore
# =================================
from fastcore.script import risinstance

# =================================
# bonsai
# =================================
from bonsai.backend import get_backend
from bonsai.optimizers import Adam
from bonsai.utils import *
from bonsai.callbacks import CSVLogger, MeanLossGraphCallback
from bonsai.visualize import teal_cmap

The engine module provides advanced functionalities for model training, including configurable training loops and evaluation functions tailored for bioinformatics applications. This module is significantly valuable when there is a need for specific workflows and pipelines that meet specific requirements. For this reason, the classes fastTrainer and visionTrainer have been created, providing tailored implementations inheriting from the Learner class. 


## Base Classes

### Backend Registry

In [ ]:
#| export

TrainerFactory = Callable[..., Any]

BACKEND_REGISTRY: Dict[str, Type["TrainerBackend"]] = {}
TRAINER_REGISTRY: Dict[str, Dict[str, TrainerFactory]] = {}



In [ ]:
#| export

def _validate_name(name: str, kind: str) -> str:
    """
    Validate and normalize a registry name.

    Parameters
    ----------
    name
        Name to evaluate.
    kind
        Type of registry entry, used in error messages.

    Returns
    -------
    str
        Stripped and lowercase name.

    Raises
    ------
    ValueError
        If ``name`` is not a non-empty string.
    """
    if not isinstance(name, str) or not name.strip():
        raise ValueError(f"{kind} name must be a non-empty string.")

    return name.strip().lower()


In [ ]:
#| export

def register_backend(
        name: str,
) -> Callable[[Type["TrainerBackend"]], Type["TrainerBackend"]]:
    """
    Register a training backend adapter.

    The decorator associates a normalized backend name with a
    :class:`TrainerBackend` subclass. Registered backends can subsequently
    be selected by name through :class:`BioTrainer`.

    Registering a backend also initializes its trainer registry, allowing
    trainer implementations to be registered with :func:`register_trainer`.

    Parameters
    ----------
    name
        Public name used to identify the backend. The name is stripped of
        leading and trailing whitespace and converted to lowercase.

    Returns
    -------
    Callable
        A class decorator that registers the backend class and returns it
        unchanged.

    Raises
    ------
    ValueError
        If ``name`` is empty or a backend with the same normalized name
        has already been registered.
    TypeError
        If the decorated object is not a subclass of
        :class:`TrainerBackend`.

    Examples
    --------
    ```python
    @register_backend("monai")
    class MonaiTrainerBackend(TrainerBackend):
        ...
    ```
    """
    backend_name = _validate_name(name, "Backend")

    def decorator(
            cls: Type["TrainerBackend"],
    ) -> Type["TrainerBackend"]:
        if not isinstance(cls, type) or not issubclass(cls, TrainerBackend):
            raise TypeError(
                "A backend must inherit from TrainerBackend."
            )

        if backend_name in BACKEND_REGISTRY:
            raise ValueError(
                f"Backend {backend_name!r} is already registered."
            )

        BACKEND_REGISTRY[backend_name] = cls
        TRAINER_REGISTRY.setdefault(backend_name, {})

        return cls

    return decorator

In [ ]:
#| export

def register_trainer(
        backend: str,
        name: str = "supervised",
) -> Callable[[TrainerFactory], TrainerFactory]:
    """
    Register a trainer implementation for a training backend.

    The decorator associates a normalized trainer name with a callable
    trainer implementation within a registered backend. The callable may
    be a trainer class, factory function, or other callable object.

    Trainer names are scoped to their backend, so the same name can be
    registered independently for different backends. For example,
    ``"supervised"`` may refer to a fastai trainer for the ``"fastai"``
    backend and a MONAI trainer for the ``"monai"`` backend.

    Parameters
    ----------
    backend
        Name of the registered backend to which the trainer belongs.
        The name is stripped of leading and trailing whitespace and
        converted to lowercase.
    name
        Public name used to identify the trainer within the backend.
        The name is normalized in the same way as ``backend``.

    Returns
    -------
    Callable
        A decorator that registers the trainer implementation and
        returns it unchanged.

    Raises
    ------
    ValueError
        If ``backend`` or ``name`` is empty, if ``backend`` has not been
        registered, or if a trainer with the same normalized name is
        already registered for the backend.
    TypeError
        If the decorated object is not callable.

    Notes
    -----
    ``register_trainer`` does not impose a specific trainer interface.
    The registered implementation is resolved by the backend and is
    expected to provide the functionality required by :class:`BioTrainer`.

    Examples
    --------
    Register a trainer class:

    ```python
    @register_trainer("fastai", "supervised")
    class FastaiTrainer:
        ...
    ```

    Register a trainer factory:

    ```python
    @register_trainer("monai", "supervised")
    def create_supervised(*args, **kwargs):
        return SupervisedTrainer(*args, **kwargs)
    ```
    """
    backend_name = _validate_name(backend, "Backend")
    trainer_name = _validate_name(name, "Trainer")

    if backend_name not in BACKEND_REGISTRY:
        raise ValueError(
            f"Cannot register trainer {trainer_name!r}: "
            f"backend {backend_name!r} is not registered."
        )

    def decorator(factory: TrainerFactory) -> TrainerFactory:
        if not callable(factory):
            raise TypeError(
                "A trainer must be a callable class or factory."
            )

        trainers = TRAINER_REGISTRY[backend_name]

        if trainer_name in trainers:
            raise ValueError(
                f"Trainer {trainer_name!r} is already registered "
                f"for backend {backend_name!r}."
            )

        trainers[trainer_name] = factory
        return factory

    return decorator

### Configuration

In [ ]:
#| export

@dataclass
class TrainerConfig:
    """
    Backend-independent configuration for :class:`BioTrainer`.

    This class defines the canonical vocabulary used by bonsai. Backend
    adapters translate these values into the corresponding native training
    framework API.

    Parameters
    ----------
    model
        Model or network to train.
    dls
        bonsai DataLoaders containing training and validation data.
    optimizer
        Optimizer instance or optimizer factory.
    loss
        Loss function used during training.
    metrics
        Metrics evaluated during training.
    val_metrics
        Metrics evaluated during validation. If ``None``, ``metrics`` are
        used for validation.
    epochs
        Number of training epochs.
    lr
        Learning rate, when applicable.
    validate
        Whether validation should be performed during training.
    valid_every
        Number of epochs between validation runs. ``1`` means validation
        after every epoch.
    device
        Training and evaluation device.
    callbacks
        Backend-independent callbacks or lifecycle handlers.
    inferer
        Optional inference strategy used during validation.
    postprocessing
        Optional post-processing applied during validation.
    csv_logger
        Whether to enable CSV logging, when supported by the backend.
    show_graph
        Whether to display training graphs, when supported by the backend.
    show_summary
        Whether to display a model summary, when supported by the backend.
    find_lr
        Whether to perform learning-rate finding before training.
    find_lr_kwargs
        Arguments passed to the learning-rate finder.
    save_dir
        Directory used for model checkpoints and other training artifacts.
    backend
        Resolved training backend selected through
        :func:`bonsai.get_backend`.
    trainer
        Training strategy implemented by the selected backend.
    backend_kwargs
        Backend-specific arguments that do not belong in the common API.

    Notes
    -----
    The names in this class are deliberately independent of native
    frameworks. For example, bonsai uses ``epochs`` rather than MONAI's
    ``max_epochs`` and ``loss`` rather than ``loss_function``.

    The active backend is selected globally with
    :func:`bonsai.set_backend`. ``TrainerConfig`` stores the resolved
    backend so that the trainer adapter can consistently use the backend
    selected when the :class:`BioTrainer` was created.

    ``val_metrics`` is intentionally separate from ``metrics`` so that
    training and validation can use different metric sets. When
    ``val_metrics`` is ``None``, the training metrics are reused for
    validation.
    """

    model: Any = None
    dls: Any = None

    optimizer: Any = None
    loss: Any = None
    metrics: Any = None
    val_metrics: Any = None

    epochs: int = 1
    lr: Optional[float] = None

    validate: bool = True
    valid_every: int = 1

    device: Any = None
    callbacks: Any = None
    val_callbacks: Any = None

    inferer: Any = None
    postprocessing: Any = None

    csv_logger: bool = False
    show_graph: bool = False
    show_summary: bool = False
    find_lr: bool = False
    find_lr_kwargs: Dict[str, Any] = field(default_factory=dict)
    save_dir: str | Path = "models"

    backend: str = "monai"
    trainer: str = "supervised"

    backend_kwargs: Dict[str, Any] = field(default_factory=dict)

In [ ]:
#| export

def _validate_trainer_config(config: TrainerConfig):
    """
    Validate backend-independent trainer configuration.

    Parameters
    ----------
    config
        Trainer configuration to evaluate.

    Raises
    ------
    ValueError
        If an invalid epoch count, validation frequency, or backend name
        is supplied.
    """
    if config.epochs < 1:
        raise ValueError("epochs must be >= 1")

    if config.valid_every < 1:
        raise ValueError("valid_every must be >= 1")

    config.backend = _validate_name(
        config.backend,
        "Backend",
    )

In [ ]:
#| export

def _prediction_activation(pred, loss_fn, act=None):
    """Use the loss activation, with native torch classification fallbacks."""
    activation = act if act is not None else getattr(loss_fn, "activation", None)
    if callable(activation):
        return activation(pred)
    if isinstance(loss_fn, torch.nn.CrossEntropyLoss):
        return pred.softmax(dim=1)
    if isinstance(loss_fn, torch.nn.NLLLoss):
        return pred.exp()
    if isinstance(loss_fn, (torch.nn.BCEWithLogitsLoss, torch.nn.MultiLabelSoftMarginLoss)):
        return pred.sigmoid()
    return pred


def _prediction_decode(pred, loss_fn):
    """Decode activated batches; unknown losses retain their predictions."""
    decode = getattr(loss_fn, "decodes", None)
    if callable(decode):
        return decode(pred)
    if isinstance(loss_fn, (torch.nn.CrossEntropyLoss, torch.nn.NLLLoss)):
        return pred.argmax(dim=1)
    if isinstance(loss_fn, (torch.nn.BCELoss, torch.nn.BCEWithLogitsLoss,
                            torch.nn.MultiLabelSoftMarginLoss)):
        return pred > 0.5
    return pred


def _metric_prediction(metric, pred, loss_fn):
    """Unmarked native metrics retain their own prediction preparation."""
    decode = getattr(metric, "_decode", None)
    if not isinstance(decode, bool):
        return pred
    if isinstance(pred, (list, tuple)):
        return type(pred)(_metric_prediction(metric, p.unsqueeze(0), loss_fn)[0] for p in pred)
    pred = _prediction_activation(pred, loss_fn)
    return _prediction_decode(pred, loss_fn) if decode else pred



In [ ]:
#| export

def _monai_metric_config(metrics, val_metrics=None, loss=None):
    """
    Build the MONAI trainer metric configuration.

    The first training metric is used as ``key_train_metric`` and the first
    validation metric is used as ``key_val_metric``. Remaining metrics are
    registered as additional metrics.

    If ``val_metrics`` is not provided, ``metrics`` is used as the validation
    metrics. If ``loss`` is provided, it is wrapped in a MONAI ``LossMetric``
    and appended to the validation metrics.

    Parameters
    ----------
    metrics : metric, sequence of metrics, or None
        Metric instance or collection of metric instances to register for
        training. The first metric is used as the key training metric.
    val_metrics : metric, sequence of metrics, or None
        Metric instance or collection of metric instances to register for
        validation. The first metric is used as the key validation metric.
        If ``None``, ``metrics`` is used as the validation metrics.
    loss : callable or None
        Loss function used to compute the validation loss. If provided, it is
        wrapped in a MONAI ``LossMetric`` and appended to the validation
        metrics.

    Returns
    -------
    dict
        MONAI trainer metric configuration. Depending on the supplied
        metrics and loss, this may contain ``key_train_metric``,
        ``key_val_metric``, ``additional_metrics``, and
        ``additional_val_metrics``.

    Notes
    -----
    Metric names are obtained from the ``name`` attribute when available.
    If ``name`` is callable, it is called to obtain the metric name.
    Otherwise, the metric class name is used with a trailing ``"Metric"``
    suffix removed.

    Each metric is wrapped in an ``IgniteMetricHandler`` using the
    ``pred`` and ``label`` outputs from the engine. If ``loss`` is provided,
    a MONAI ``LossMetric`` is appended to the validation metrics to track
    validation loss using raw engine outputs. Metrics with ``_decode`` use
    loss activation and, when True, loss decoding; unmarked native metrics
    receive raw outputs.
    """

    def metric_name(metric):
        name = getattr(
            metric,
            "name",
            metric.__class__.__name__.removesuffix("Metric"),
        )
        return name() if callable(name) else name

    def wrap_handler(metric):
        def prepare(output):
            pred, target = from_engine(["pred", "label"])(output)
            if not isinstance(metric, LossMetric):
                pred = _metric_prediction(metric, pred, loss)
            return pred, target

        return IgniteMetricHandler(metric_fn=metric, output_transform=prepare)

    def as_list(metrics):
        if metrics is None:
            return None
        return list(metrics) if isinstance(metrics, (list, tuple)) else [metrics]

    metrics = as_list(metrics)
    val_metrics = as_list(val_metrics)

    if val_metrics is None:
        val_metrics = list(metrics) if metrics else []

    # Append validation loss as a MONAI LossMetric.
    if loss is not None:
        val_metrics.append(
            LossMetric(loss_fn=loss)
        )

    values = {}

    if metrics:
        values["key_train_metric"] = {
            metric_name(metrics[0]): wrap_handler(metrics[0]),
        }

        if len(metrics) > 1:
            values["additional_metrics"] = {
                metric_name(metric): wrap_handler(metric)
                for metric in metrics[1:]
            }

    if val_metrics:
        values["key_val_metric"] = {
            metric_name(val_metrics[0]): wrap_handler(val_metrics[0]),
        }

        if len(val_metrics) > 1:
            values["additional_val_metrics"] = {
                metric_name(metric): wrap_handler(metric)
                for metric in val_metrics[1:]
            }

    return values

In [ ]:
#| export

def _native_config(self) -> Dict[str, Any]:
    """
    Return the trainer configuration using native argument names.

    Returns
    -------
    dict
        Configuration arguments mapped to the names expected by the native
        trainer. Metrics are normalized to the dictionary format expected by
        MONAI. Explicit values provided through ``backend_kwargs`` override
        the generated configuration.
    """
    cfg = self.config

    values = {
        "network": cfg.model,
        "train_data_loader": cfg.dls.train,
        "optimizer": cfg.optimizer,
        "loss_function": cfg.loss,
        "train_handlers": cfg.callbacks,
        "device": cfg.device,
        "inferer": cfg.inferer,
        "postprocessing": cfg.postprocessing,
        "max_epochs": cfg.epochs,
        "val_data_loader": cfg.dls.valid,
        "val_handlers": cfg.val_callbacks,
    }

    values.update(_monai_metric_config(cfg.metrics, cfg.val_metrics, cfg.loss))
    values.update(cfg.backend_kwargs)

    return values

In [ ]:
#| export
def _translate_config(
        self,
        target=None,
        *,
        drop_none: bool = True,
) -> Dict[str, Any]:
    """
    Translate the trainer configuration to arguments accepted by the
    native trainer.

    Parameters
    ----------
    target
        Native trainer callable or class that will receive the arguments.
        If omitted, ``trainer_cls`` is used.
    drop_none
        If ``True``, arguments whose value is ``None`` are omitted.

    Returns
    -------
    dict
        Keyword arguments compatible with the native trainer.

    Notes
    -----
    The backend-specific :meth:`_native_config` method maps the canonical
    trainer configuration to native argument names. The resulting arguments
    are then filtered against the native trainer signature. Arguments supplied
    through ``backend_kwargs`` are included as framework-specific overrides.
    """
    target = target or self.trainer_cls

    values = self._native_config()

    if drop_none:
        values = {
            name: value
            for name, value in values.items()
            if value is not None
        }

    try:
        signature = inspect.signature(target)
    except (TypeError, ValueError):
        return values

    if any(
            parameter.kind == inspect.Parameter.VAR_KEYWORD
            for parameter in signature.parameters.values()
    ):
        return values

    allowed = set(signature.parameters)

    return {
        name: value
        for name, value in values.items()
        if name in allowed
    }

In [ ]:
#| export
def _get_trainer(self):
    """
    Return the trainer factory registered for the current backend and
    trainer type.
    """
    backend_trainers = TRAINER_REGISTRY.get(
        self.config.backend,
        {},
    )

    trainer = backend_trainers.get(
        self.config.trainer,
    )

    if trainer is None:
        available = ", ".join(
            sorted(backend_trainers)
        )

        raise ValueError(
            f"Unknown trainer {self.config.trainer!r} for "
            f"backend {self.config.backend!r}. "
            f"Available trainers: {available or 'none'}"
        )

    return trainer

In [ ]:
#| export

def _build_trainer(self):
    """
    Build the trainer selected by ``config.trainer``.

    Returns
    -------
    Any
        The constructed trainer.
    """
    trainer_factory = self._get_trainer()

    kwargs = self._translate_config(trainer_factory, drop_none=True)

    return trainer_factory(**kwargs)

### Trainer Backend

In [ ]:
#| export

class TrainerBackend:
    """
    Base class for bonsai training backends.

    A backend translates the canonical :class:`TrainerConfig` into the
    native API of a particular training framework.

    Backend implementations are responsible for:

    - translating common argument names;
    - adapting bonsai objects to native objects;
    - constructing the native trainer;
    - implementing validation;
    - implementing backend-specific training lifecycle logic.

    Parameters
    ----------
    config
        Canonical trainer configuration.

    Notes
    -----
    Framework-specific concepts should remain inside the backend. For
    example, MONAI/Ignite events should not become part of the public
    ``BioTrainer`` API.

    The native trainer is constructed eagerly during backend initialization.
    This ensures that ``trainer``, ``evaluate``, ``predict``, and other
    lifecycle methods can operate on an already-configured native trainer.
    """

    config_map: Mapping[str, Any] = {}
    trainer_cls: Any = None

    def __init__(self, config: TrainerConfig):
        self.config = config
        self.evaluator = None
        self.trainer = self._build_trainer()

    def show_results(self, dl=None, max_n=9, **kwargs):
        """Display predictions using the native trainer's visualization."""
        return self.trainer.show_results(dl=dl, max_n=max_n, **kwargs)

    @property
    def recorder(self):
        """Return the backend training recorder, when available."""
        return None


TrainerBackend._validate_trainer_config = _validate_trainer_config
TrainerBackend._native_config = _native_config
TrainerBackend._translate_config = _translate_config
TrainerBackend._get_trainer = _get_trainer
TrainerBackend._build_trainer = _build_trainer

In [ ]:
#| export

def _trainer_backend_fit(self):
    raise NotImplementedError(
        f"{self.__class__.__name__} does not implement fit()."
    )


def _trainer_backend_validate(self):
    raise NotImplementedError(
        f"{self.__class__.__name__} does not implement evaluate()."
    )


def _trainer_backend_predict(self, *args, **kwargs):
    raise NotImplementedError(
        f"{self.__class__.__name__} does not implement predict()."
    )


TrainerBackend.fit = _trainer_backend_fit
TrainerBackend.validate = _trainer_backend_validate
TrainerBackend.predict = _trainer_backend_predict

### BioTrainer

In [ ]:
#| export

def _instantiate_component(component, kwargs=None, *args):
    """
    Instantiate a component when it is a class.

    Already-instantiated callables are returned unchanged. This allows users
    to pass either a class/factory or an already-configured callable object.

    Parameters
    ----------
    component
        Component class or already-instantiated callable.
    kwargs
        Arguments used when instantiating a class.
    *args
        Positional arguments passed to the constructor.

    Returns
    -------
    Any
        Instantiated component or the original callable.

    Notes
    -----
    Backend selection is handled internally by the component through the
    globally configured bonsai backend. The helper therefore does not
    inject or override a ``backend`` argument.
    """
    if component is None:
        return None

    # An already-created loss, metric, or factory is used as-is.
    if not inspect.isclass(component):
        return component

    kwargs = dict(kwargs or {})

    return component(*args, **kwargs)


In [ ]:
#| export

def _instantiate_components(components, components_kwargs=None):
    """
    Instantiate component classes while preserving already-created callables.

    Parameters
    ----------
    components : component, sequence of components, or None
        Component classes or already-created component instances.
    components_kwargs : dict, sequence of dict, or None
        Keyword arguments passed to the components. A single dictionary is
        shared by all components. A sequence of dictionaries provides
        per-component keyword arguments in the same order as ``components``.

    Returns
    -------
    list or None
        Instantiated components, or ``None`` if ``components`` is ``None``.

    Raises
    ------
    ValueError
        If a sequence of ``components_kwargs`` does not have the same length
        as ``components``.
    TypeError
        If ``components_kwargs`` is neither a dictionary, a sequence of
        dictionaries, nor ``None``.
    """
    if components is None:
        return None

    if not isinstance(components, (list, tuple)):
        components = [components]

    if components_kwargs is None:
        kwargs = [{} for _ in components]
    elif isinstance(components_kwargs, dict):
        kwargs = [components_kwargs] * len(components)
    elif isinstance(components_kwargs, (list, tuple)):
        if len(components_kwargs) != len(components):
            raise ValueError(
                "`components_kwargs` must have the same length as `components`."
            )
        kwargs = components_kwargs
    else:
        raise TypeError(
            "`components_kwargs` must be a dict, list/tuple of dicts, or None."
        )

    return [
        _instantiate_component(component, component_kwargs)
        for component, component_kwargs in zip(components, kwargs)
    ]

In [ ]:
#| export

class BioTrainer:
    """
    Backend-independent training interface for bonsai.

    ``BioTrainer`` exposes one canonical set of training arguments and
    delegates framework-specific implementation to a registered
    :class:`TrainerBackend`.

    The training backend is selected globally with
    :func:`bonsai.set_backend` and is shared automatically by the
    trainer, optimizer, loss, and metrics. The ``trainer`` argument
    selects the training strategy implemented by the active backend.

    Components such as losses, metrics, and optimizers can be supplied
    either as classes, in which case ``BioTrainer`` constructs them using
    the active backend, or as already-instantiated callables, in which
    case they are used unchanged.

    Parameters
    ----------
    model
        Model or network to train.
    dls
        bonsai DataLoaders containing training and validation data.
    optimizer
        Optimizer class or already-instantiated optimizer. When a class is
        provided, it is instantiated with the model parameters and
        ``optimizer_kwargs``.
    optimizer_kwargs
        Additional arguments used to instantiate ``optimizer`` when it is
        provided as a class.
    loss
        Loss class or already-instantiated callable. When a class is
        provided, it is instantiated using ``loss_kwargs``.
    loss_kwargs
        Additional arguments used to instantiate ``loss`` when it is
        provided as a class.
    metrics
        Metric class, callable, or sequence of metric classes/callables
        evaluated during training. Metric classes are instantiated using
        ``metrics_kwargs``.
    metrics_kwargs
        Additional arguments used to instantiate training metrics. A
        dictionary is shared by all metrics, while a sequence of
        dictionaries provides arguments for each metric individually.
    val_metrics
        Metric class, callable, or sequence of metric classes/callables
        evaluated during validation. If ``None``, ``metrics`` are used.
    val_metrics_kwargs
        Additional arguments used to instantiate validation metrics. A
        dictionary is shared by all validation metrics, while a sequence
        of dictionaries provides arguments for each metric individually.
        If ``None`` and ``val_metrics`` is ``None``, ``metrics_kwargs`` are
        used.
    epochs
        Number of training epochs.
    lr
        Learning rate.
    validate
        Whether to perform validation during training.
    valid_every
        Number of epochs between validation runs.
    device
        Training device. For MONAI, the model is moved here before
        constructing an optimizer class.
    callbacks
        Backend-independent callbacks.
    inferer
        Optional inference strategy.
    postprocessing
        Optional validation post-processing.
    csv_logger
        Whether to enable CSV logging when supported by the selected
        trainer.
    show_graph
        Whether to display training graphs when supported by the selected
        trainer.
    show_summary
        Whether to display a model summary when supported by the selected
        trainer.
    find_lr
        Whether to perform learning-rate finding when supported by the
        selected trainer.
    find_lr_kwargs
        Additional arguments passed to the learning-rate finder.
    save_dir
        Directory used for saved models and training artifacts.
    trainer
        Training strategy registered for the active backend, such as
        ``"supervised"`` or ``"gan"``.
    backend_kwargs
        Optional backend-specific arguments that are not part of the common
        ``BioTrainer`` interface.

    Notes
    -----
    The backend is selected globally before creating the trainer::

        bonsai.set_backend("fastai")

        trainer = BioTrainer(
            model=model,
            dls=dls,
            optimizer=Adam,
            loss=CrossEntropyLoss,
            metrics=[AccuracyMetric],
            epochs=2,
            lr=1e-3,
        )

    Changing the global backend affects subsequently constructed
    components and trainers. Already-instantiated components are not
    reconstructed.

    ``BioTrainer`` deliberately does not expose native framework names
    such as MONAI's ``max_epochs`` or ``loss_function``. Backend adapters
    perform that translation.

    ``trainer`` and the global backend are independent. A backend can
    provide multiple trainer implementations, and the same trainer name
    can be implemented by multiple backends.
    """

    def __init__(
            self,
            model=None,
            dls=None,
            *,
            optimizer=None,
            optimizer_kwargs=None,
            loss=None,
            loss_kwargs=None,
            metrics=None,
            metrics_kwargs=None,
            val_metrics=None,
            val_metrics_kwargs=None,
            epochs=1,
            lr=None,
            validate=True,
            valid_every=1,
            device=None,
            callbacks=None,
            callbacks_kwargs=None,
            val_callbacks=None,
            val_callbacks_kwargs=None,
            inferer=None,
            postprocessing=None,
            csv_logger=False,
            show_graph=False,
            show_summary=False,
            find_lr=False,
            find_lr_kwargs=None,
            save_dir="models",
            trainer="supervised",
            backend_kwargs=None,
    ):
        backend = get_backend()

        optimizer_kwargs = dict(optimizer_kwargs or {})
        loss_kwargs = dict(loss_kwargs or {})
        metrics_kwargs = dict(metrics_kwargs or {})

        if val_metrics is None:
            val_metrics = metrics
            if val_metrics_kwargs is None:
                val_metrics_kwargs = metrics_kwargs

        # Resolve framework-independent components before constructing the
        # backend. Every component uses the same globally selected backend.
        resolved_loss = _instantiate_component(
            loss,
            loss_kwargs,
        )

        resolved_metrics = _instantiate_components(
            metrics,
            metrics_kwargs,
        )

        resolved_val_metrics = _instantiate_components(
            val_metrics,
            val_metrics_kwargs,
        )

        if backend == "monai" and model is not None:
            training_device = (backend_kwargs or {}).get("device", device)
            if training_device is not None:
                # Move before optimizer construction so it uses the final parameters.
                model = model.to(training_device)

        resolved_optimizer = self._resolve_optimizer(
            optimizer,
            optimizer_kwargs,
            model,
        )

        resolved_callbacks = _instantiate_components(
            callbacks,
            callbacks_kwargs,
        )

        resolved_val_callbacks = _instantiate_components(
            val_callbacks,
            val_callbacks_kwargs,
        )

        self.config = TrainerConfig(
            model=model,
            dls=dls,
            optimizer=resolved_optimizer,
            loss=resolved_loss,
            metrics=resolved_metrics,
            val_metrics=resolved_val_metrics,
            epochs=epochs,
            lr=lr,
            validate=validate,
            valid_every=valid_every,
            device=device,
            callbacks=resolved_callbacks,
            val_callbacks=resolved_val_callbacks,
            inferer=inferer,
            postprocessing=postprocessing,
            csv_logger=csv_logger,
            show_graph=show_graph,
            show_summary=show_summary,
            find_lr=find_lr,
            find_lr_kwargs=find_lr_kwargs or {},
            save_dir=save_dir,
            backend=backend,
            trainer=trainer,
            backend_kwargs=backend_kwargs or {},
        )

        _validate_trainer_config(self.config)

        self.backend = self._build_backend()

    @staticmethod
    def _resolve_optimizer(optimizer, kwargs, model):
        """
        Instantiate an optimizer class or return an existing optimizer.

        Optimizer classes are instantiated with ``model.parameters()``.
        Already-instantiated optimizers or factories are returned unchanged.
        """
        if optimizer is None:
            return None

        if not inspect.isclass(optimizer):
            return optimizer

        if model is None:
            raise ValueError(
                "A model is required when 'optimizer' is provided as a class."
            )

        return optimizer(
            model.parameters(),
            **dict(kwargs),
        )

    def _build_backend(self):
        """
        Instantiate the trainer backend selected by the global backend.

        Returns
        -------
        TrainerBackend
            Backend responsible for implementing the selected training
            strategy.
        """
        backend_cls = BACKEND_REGISTRY.get(self.config.backend)

        if backend_cls is None:
            available = ", ".join(
                sorted(BACKEND_REGISTRY)
            )

            raise ValueError(
                f"Unknown training backend "
                f"'{self.config.backend}'. "
                f"Available backends: {available}"
            )

        return backend_cls(self.config)

    def fit(self):
        """
        Train the model using the selected backend and trainer.

        Returns
        -------
        BioTrainer
            The trainer instance.
        """
        return self.backend.fit()

    if TYPE_CHECKING:
        # PyCharm needs a static target for MONAI hover documentation.
        from bonsai.engines import _monai_evaluate as evaluate
    else:
        @property
        def evaluate(self):
            """Return the backend's bound evaluation method and its signature."""
            return self.backend.evaluate

    def predict(self, *args, **kwargs):
        """
        Run inference using the selected backend.

        Parameters
        ----------
        *args
            Positional backend-specific arguments.
        **kwargs
            Keyword backend-specific arguments.

        Returns
        -------
        Any
            Backend-specific prediction result.
        """
        return self.backend.predict(*args, **kwargs)

    def show_results(self, dl=None, max_n=9, **kwargs):
        """Display inputs, targets, and decoded predictions from one batch.

        Uses the validation loader unless `dl` is supplied. `max_n` limits
        the number of displayed samples; `kwargs` are plotting options.
        MONAI uses its configured evaluator; fastai uses the native learner.
        Returns the backend's visualization result.
        """
        return self.backend.show_results(dl=dl, max_n=max_n, **kwargs)

    @property
    def trainer_instance(self):
        """
        Return the instantiated trainer.

        Returns
        -------
        Any
            Native or bonsai trainer instance.
        """
        return self.backend.trainer

    @property
    def evaluator_instance(self):
        """
        Return the instantiated validation evaluator, when applicable.

        Returns
        -------
        Any or None
            Backend evaluator, or ``None`` when validation is integrated
            directly into the trainer.
        """
        return self.backend.evaluator

    @property
    def recorder(self):
        """
        Return the training recorder, when supported by the active backend.

        Returns
        -------
        Any or None
            Backend-specific training recorder, or ``None`` when the active
            backend does not provide one.
        """
        return self.backend.recorder

---

## Evaluation Utils

In [ ]:
#| export

def _evaluation_metrics(metrics):
    """Normalize metric specifications and isolate stateful metrics."""
    from collections.abc import Mapping
    from copy import copy

    if metrics is None:
        return {}

    if isinstance(metrics, Mapping):
        items = list(metrics.items())
    else:
        values = (
            [metrics]
            if callable(metrics) or hasattr(metrics, "reset")
            else list(metrics)
        )
        items = []
        for metric in values:
            name = getattr(metric, "name", None)
            if callable(name):
                name = name()
            name = name or getattr(
                metric, "__name__", type(metric).__name__
            )
            items.append((name, metric))

    result = {}

    for name, metric in items:
        if name in result:
            raise ValueError(
                f"Duplicate metric name {name!r}; "
                "use a mapping with unique names."
            )

        if isinstance(metric, type):
            instance = metric()
        else:
            instance = copy(metric)

        result[name] = instance

    return result


def _evaluation_detach(value):
    if isinstance(value, torch.Tensor):
        return value.detach().cpu()
    if isinstance(value, dict):
        return {k: _evaluation_detach(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return type(value)(_evaluation_detach(v) for v in value)
    return value

def _display_evaluation_statistics(result):
    """Show one distribution and statistics table per collected loss or metric."""
    import warnings
    from bonsai.visualize import plot_stats, display_statistics_table

    distributions = []
    if "losses" in result:
        distributions.append(("Loss", result["losses"]))
    distributions.extend(result.get("metrics", {}).items())

    for name, values in distributions:
        arrays = [
            value.detach().cpu().float().numpy().ravel()
            if isinstance(value, torch.Tensor)
            else np.asarray(value, dtype=float).ravel()
            for value in values
        ]
        data = np.concatenate(arrays) if arrays else np.array([], dtype=float)
        data = data[np.isfinite(data)]
        if not data.size:
            warnings.warn(
                f"No finite values to show_stats for {name!r}.",
                stacklevel=2,
            )
            continue

        stats = plot_stats(
            data, fn_name=name,
            show_kde=data.size > 1 and np.ptp(data) > 0,
        )
        display_statistics_table(stats, fn_name=name)

In [ ]:
#| export

def _classification_data(output, vocab=None, threshold=0.5):
    """Convert evaluation outputs to single-label class IDs and class names.

    Accepts CPU/GPU tensors, NumPy arrays, or MONAI lists of samples.
    Uses decoded predictions when available; otherwise uses argmax for
    class scores or threshold for binary probabilities. For binary logits,
    apply sigmoid through evaluate(act=...) or pass threshold=0 here.
    """

    def as_array(values):
        if isinstance(values, torch.Tensor):
            return values.detach().cpu().numpy()
        if isinstance(values, (list, tuple)):
            return np.asarray([
                v.detach().cpu().numpy() if isinstance(v, torch.Tensor) else v
                for v in values
            ])
        return np.asarray(values)

    if output.get("targs") is None:
        raise ValueError("Classification reports require targs in the output dictionary.")
    source = output.get("decoded")
    decoded = source is not None
    if source is None:
        source = output.get("preds")
    if source is None:
        raise ValueError("Classification reports require preds or decoded in the output dictionary.")

    pred, target = as_array(source), as_array(output["targs"])
    if pred.ndim == 0 or target.ndim == 0 or not len(target):
        raise ValueError("Classification reports require non-empty sample arrays.")
    class_count = 0
    if target.ndim == 2 and target.shape[1] > 1:
        if not (np.isin(target, [0, 1]).all() and (target.sum(axis=1) == 1).all()):
            raise ValueError("Expected single-label or one-hot targets; multilabel targets are unsupported.")
        class_count = target.shape[1]
        target = target.argmax(axis=1)
    elif target.ndim == 2 and target.shape[1] == 1:
        target = target[:, 0]
    if pred.ndim == 2 and pred.shape[1] > 1:
        class_count = max(class_count, pred.shape[1])
        pred = pred.argmax(axis=1)
    else:
        if pred.ndim == 2 and pred.shape[1] == 1:
            pred = pred[:, 0]
        if not decoded and np.issubdtype(pred.dtype, np.floating):
            pred = (pred >= threshold).astype(int)
            class_count = max(class_count, 2)
    if pred.ndim != 1 or target.ndim != 1 or len(pred) != len(target):
        raise ValueError("Expected one classification prediction and target per sample.")
    for values in (pred, target):
        if not np.isfinite(values).all() or (values < 0).any() or not np.equal(values, np.floor(values)).all():
            raise ValueError("Class IDs must be finite non-negative integers.")
    pred, target = pred.astype(int), target.astype(int)
    class_count = max(class_count, int(max(pred.max(), target.max())) + 1)
    names = list(vocab) if vocab is not None else list(range(class_count))
    if len(names) < class_count:
        raise ValueError("vocab does not contain all observed classes.")
    return target, pred, np.arange(len(names)), names


def confusion_matrix(output, vocab=None, threshold=0.5):
    """Return a confusion matrix from evaluation outputs (rows are actual classes)."""
    from sklearn.metrics import confusion_matrix
    target, pred, labels, _ = _classification_data(output, vocab, threshold)
    return confusion_matrix(target, pred, labels=labels)


def plot_confusion_matrix(
        output, vocab=None, normalize=True, title="Confusion matrix",
        cmap="Blues", norm_dec=2, plot_txt=True, threshold=0.5, **kwargs,
):
    """Display a confusion matrix from evaluation outputs and return its axes."""
    cm = confusion_matrix(output, vocab, threshold)
    _, _, _, names = _classification_data(output, vocab, threshold)
    if normalize:
        totals = cm.sum(axis=1, keepdims=True)
        cm = np.divide(cm, totals, out=np.zeros_like(cm, dtype=float), where=totals != 0)
    fig, ax = plt.subplots(**kwargs)
    ax.set_frame_on(False)
    ax.imshow(cm, interpolation="nearest", cmap=cmap)
    ax.set(
        title=title, xlabel="Predicted", ylabel="Actual",
        xticks=np.arange(len(names)), yticks=np.arange(len(names)),
        xticklabels=names, yticklabels=names,
    )
    ax.tick_params(axis="x", labelrotation=90)
    if plot_txt:
        for i, j in np.ndindex(cm.shape):
            text = f"{cm[i, j]:.{norm_dec}f}" if normalize else str(cm[i, j])
            ax.text(j, i, text, ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black")
    ax.grid(False)
    fig.tight_layout()
    plt.show()
    return ax


def _most_confused(output, vocab=None, min_val=1, threshold=0.5, n_most_confused=5):
    """Return up to n_most_confused errors by descending count; n_most_confused=None returns all qualifying pairs."""
    cm = confusion_matrix(output, vocab, threshold)
    _, _, _, names = _classification_data(output, vocab, threshold)
    errors = [
        (names[i], names[j], int(cm[i, j]))
        for i, j in np.ndindex(cm.shape)
        if i != j and cm[i, j] >= min_val
    ]
    return sorted(errors, key=lambda item: item[2], reverse=True)[:n_most_confused]


def _skm_classification_report(output, vocab=None, threshold=0.5, output_dict=False):
    """Return sklearn's classification report from evaluation outputs."""
    from sklearn.metrics import classification_report
    target, pred, labels, names = _classification_data(output, vocab, threshold)
    return classification_report(
        target, pred, labels=labels, target_names=[str(name) for name in names],
        output_dict=output_dict, zero_division=0,
    )


In [ ]:
#| export

def classification_report(output, vocab=None, threshold=0.5, min_val=1, n_most_confused=5, **plot_kwargs):
    """Display all classification diagnostics and return their computed values."""

    from IPython.display import display

    cm = confusion_matrix(output, vocab, threshold)
    plot_confusion_matrix(output, vocab=vocab, threshold=threshold, **plot_kwargs)
    confused = _most_confused(output, vocab=vocab, min_val=min_val, threshold=threshold, n_most_confused=n_most_confused)
    confused_df = pd.DataFrame(confused, columns=["Actual", "Predicted", "Count"])
    report = _skm_classification_report(
        output, vocab=vocab, threshold=threshold, output_dict=True,
    )
    rows = {}
    for name, values in report.items():
        rows[name] = (
            {"precision": np.nan, "recall": np.nan, "f1-score": values,
             "support": report["macro avg"]["support"]}
            if name == "accuracy" else values
        )
    report_df = pd.DataFrame.from_dict(rows, orient="index")
    report_df.index.name = "Class / summary"
    report_df = report_df.reset_index()

    table_styles = [
        {"selector": "caption", "props": [
            ("caption-side", "top"), ("text-align", "left"),
            ("background-color", "transparent !important"),
            ("font-size", "1.1em"), ("font-weight", "600"),
            ("color", "#008080"), ("padding", "8px 0"),
        ]},
        {"selector": "th", "props": [
            ("background-color", "transparent !important"),
            ("color", "#285454"),
            ("padding", "8px 12px"), ("text-align", "left"),
            ("border-bottom", "2px solid #008080"),
        ]},
        {"selector": "td", "props": [
            ("padding", "8px 12px"),  #("border-bottom", "1px solid #e3e8e8"),
        ]},
        {"selector": "tbody tr:hover", "props": [("background-color", "#f2f7f7")]},
    ]
    display(
        confused_df.style.set_caption("Most confused")
        .hide(axis="index").format({"Count": "{:,.0f}"})
        .set_table_styles(table_styles)
    )
    display(
        report_df.style.set_caption("Classification report")
        .hide(axis="index")
        .format({"precision": "{:.3f}", "recall": "{:.3f}",
                 "f1-score": "{:.3f}", "support": "{:,.0f}"}, na_rep="—")
        .set_table_styles(table_styles)
    )
    return {
        "confusion_matrix": cm,
        "most_confused": confused,
        "report": report,
    }

In [ ]:
#| export

def _report_table(pdf, frame, title):
    """Write a displayed pandas table to paginated PDF pages."""
    from matplotlib.figure import Figure
    from textwrap import fill

    if not isinstance(frame.index, pd.RangeIndex) or frame.index.name is not None:
        frame = frame.reset_index()
    for start in range(0, max(1, len(frame)), 25):
        page = frame.iloc[start:start + 25]
        fig = Figure(figsize=(11.7, 8.3))
        ax = fig.subplots()
        ax.axis("off")
        ax.set_title(str(title), loc="left", color="teal", pad=20)
        if page.empty:
            ax.text(0, 0.9, "No rows", transform=ax.transAxes)
        else:
            rows = [
                [fill(f"{value:.4g}" if isinstance(value, (float, np.floating))
                      else str(value), width=32) for value in row]
                for row in page.itertuples(index=False, name=None)
            ]
            table = ax.table(cellText=rows, colLabels=[str(c) for c in page.columns],
                             cellLoc="left", loc="upper center")
            table.auto_set_font_size(False)
            table.set_fontsize(9)
            table.scale(1, 2)
            for col in range(len(page.columns)):
                table[0, col].set_facecolor("#d9eeee")
        pdf.savefig(fig, bbox_inches="tight")
        fig.clear()


@contextmanager
def _evaluation_report(save_report):
    """Capture evaluation figures and pandas tables while displaying them."""
    if save_report is None:
        yield
        return

    from tempfile import NamedTemporaryFile
    from unittest.mock import patch
    from matplotlib.backends.backend_pdf import PdfPages
    from matplotlib.figure import Figure
    from pandas.io.formats.style import Styler
    import IPython.display as ipy_display
    import bonsai.visualize as visualizations

    path = Path(save_report)
    if path.suffix.lower() != ".pdf":
        raise ValueError("save_report must be a path ending in .pdf.")
    path.parent.mkdir(parents=True, exist_ok=True)
    seen = {plt.figure(number) for number in plt.get_fignums()}
    original_show, original_display = plt.show, ipy_display.display
    with NamedTemporaryFile(dir=path.parent, suffix=".pdf", delete=False) as tmp:
        temporary = Path(tmp.name)
    try:
        with PdfPages(temporary) as pdf:
            def save_figures():
                for number in plt.get_fignums():
                    fig = plt.figure(number)
                    if fig not in seen:
                        pdf.savefig(fig, bbox_inches="tight")
                        seen.add(fig)

            def show(*args, **kwargs):
                save_figures()
                return original_show(*args, **kwargs)

            def display(*objects, **kwargs):
                save_figures()
                for obj in objects:
                    if isinstance(obj, Styler):
                        _report_table(pdf, obj.data, obj.caption or "Evaluation table")
                    elif isinstance(obj, pd.DataFrame):
                        _report_table(pdf, obj, obj.index.name or "Evaluation statistics")
                return original_display(*objects, **kwargs)

            with patch.object(plt, "show", show), patch.object(ipy_display, "display", display), \
                    patch.object(visualizations, "display", display):
                yield
                save_figures()
            if pdf.get_pagecount() == 0:
                fig = Figure(figsize=(8.3, 11.7))
                fig.text(0.1, 0.9, "No evaluation visual output was selected.")
                pdf.savefig(fig)
        temporary.replace(path)
    finally:
        temporary.unlink(missing_ok=True)


In [ ]:
#| export
def _display_evaluation_outputs(
        result, preview, show_stats, print_report, save_report,
        report_data=None, **report_kwargs,
):
    """Share report capture, statistics, and classification displays."""
    with _evaluation_report(save_report):
        if preview is not None:
            preview()
        if show_stats:
            _display_evaluation_statistics(result)
        if print_report:
            result["classification_report"] = classification_report(
                result if report_data is None else report_data, **report_kwargs,
            )


In [ ]:
#| hide
import nbdev;

nbdev.nbdev_export()